## INSTALL requiremnts

In [1]:
pip install torch torchvision timm scikit-learn matplotlib seaborn

## Import Libraries

In [2]:

import os
import torch
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader
import torch.nn as nn
import torch.optim as optim

from sklearn.metrics import (
    confusion_matrix, classification_report,
    roc_curve, auc, f1_score,
    precision_score, recall_score,
    mean_squared_error, cohen_kappa_score
)

## Dataset

In [3]:

TRAIN_DIR = "/content/drive/MyDrive/Knee_OADataset/224ClaheOAI4class/train"
TEST_DIR  = "/content/drive/MyDrive/Knee_OADataset/224ClaheOAI4class/test"

IMG_SIZE = 224
BATCH_SIZE = 64
EPOCHS = 80
LR = 1e-3
NUM_CLASSES = 4
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

CLASS_NAMES = ["Healthy", "Mild", "Moderate", "Severe"]

# -------------------------
# DATA AUGMENTATION
# -------------------------
train_transforms = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(20),
    transforms.RandomResizedCrop(IMG_SIZE, scale=(0.9,1.0)),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize([0.5],[0.5])
])

test_transforms = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.5],[0.5])
])

In [5]:
# -------------------------
# DATA LOADERS
# -------------------------
train_dataset = datasets.ImageFolder(TRAIN_DIR, transform=train_transforms)
test_dataset  = datasets.ImageFolder(TEST_DIR, transform=test_transforms)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE,
                          shuffle=True, num_workers=2)

test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE,
                         shuffle=False, num_workers=2)

## DenseNet169 Model

In [6]:
# -------------------------
# MODEL
# -------------------------
class KneeOA_DenseNet169(nn.Module):
    def __init__(self):
        super().__init__()

        self.backbone = models.densenet169(pretrained=True)
        in_features = self.backbone.classifier.in_features

        self.backbone.classifier = nn.Identity()

        self.classifier = nn.Sequential(
            nn.Linear(in_features, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.Dropout(0.4),

            nn.Linear(256, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Dropout(0.3),

            nn.Linear(128, NUM_CLASSES)
        )

    def forward(self, x):
        x = self.backbone(x)
        x = self.classifier(x)
        return x

model = KneeOA_DenseNet169().to(DEVICE)

/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=DenseNet169_Weights.IMAGENET1K_V1`. You can also use `weights=DenseNet169_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Downloading: "https://download.pytorch.org/models/densenet169-b2777c0a.pth" to /root/.cache/torch/hub/checkpoints/densenet169-b2777c0a.pth


100%|██████████| 54.7M/54.7M [00:00<00:00, 68.1MB/s]


In [10]:
pip install torchinfo


In [11]:
from torchinfo import summary

summary(
    model,
    input_size=(1, 3, 224, 224),   # (batch, channels, height, width)
    col_names=["input_size", "output_size", "num_params"],
    depth=4
)

Layer (type:depth-idx)                        Input Shape               Output Shape              Param #
KneeOA_DenseNet169                            [1, 3, 224, 224]          [1, 4]                    --
├─DenseNet: 1-1                               [1, 3, 224, 224]          [1, 1664]                 --
│    └─Sequential: 2-1                        [1, 3, 224, 224]          [1, 1664, 7, 7]           --
│    │    └─Conv2d: 3-1                       [1, 3, 224, 224]          [1, 64, 112, 112]         9,408
│    │    └─BatchNorm2d: 3-2                  [1, 64, 112, 112]         [1, 64, 112, 112]         128
│    │    └─ReLU: 3-3                         [1, 64, 112, 112]         [1, 64, 112, 112]         --
│    │    └─MaxPool2d: 3-4                    [1, 64, 112, 112]         [1, 64, 56, 56]           --
│    │    └─_DenseBlock: 3-5                  [1, 64, 56, 56]           [1, 256, 56, 56]          --
│    │    │    └─_DenseLayer: 4-1             [1, 64, 56, 56]           [1, 32, 56

## Train Model

In [ ]:
# -------------------------
# LOSS & OPTIMIZER
# -------------------------
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=LR)

# -------------------------
# TRAINING LOOP
# -------------------------
train_acc, val_acc = [], []
train_loss, val_loss = [], []

for epoch in range(EPOCHS):
    model.train()
    running_loss, correct = 0, 0

    for images, labels in train_loader:
        images, labels = images.to(DEVICE), labels.to(DEVICE)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        preds = outputs.argmax(1)
        correct += (preds == labels).sum().item()

    train_loss.append(running_loss/len(train_loader))
    train_acc.append(correct/len(train_dataset))

    # ------------------
    model.eval()
    running_loss, correct = 0, 0

    with torch.no_grad():
        for images, labels in test_loader:
            images, labels = images.to(DEVICE), labels.to(DEVICE)
            outputs = model(images)
            loss = criterion(outputs, labels)

            running_loss += loss.item()
            preds = outputs.argmax(1)
            correct += (preds == labels).sum().item()

    val_loss.append(running_loss/len(test_loader))
    val_acc.append(correct/len(test_dataset))

    print(f"Epoch {epoch+1}/{EPOCHS} | "
          f"Train Acc: {train_acc[-1]:.4f} | "
          f"Val Acc: {val_acc[-1]:.4f}")

# -------------------------
# SAVE MODEL
# -------------------------
torch.save(model.state_dict(), "knee_oa_densenet169.pth")

## METRICS


In [ ]:

precision = precision_score(y_true, y_pred, average="weighted")
recall    = recall_score(y_true, y_pred, average="weighted")
f1        = f1_score(y_true, y_pred, average="weighted")
mse       = mean_squared_error(y_true, y_pred)
kappa     = cohen_kappa_score(y_true, y_pred)

specificity = np.mean(
    np.diag(cm) / (np.sum(cm,axis=1) + 1e-6)
)

print("\nPrecision:",precision)
print("Recall:",recall)
print("F1-score:",f1)
print("Specificity:",specificity)
print("MSE:",mse)
print("Kappa:",kappa)

print("\nClassification Report\n")
print(classification_report(y_true, y_pred, target_names=CLASS_NAMES))

## PLOT ACCURACY & LOSS


In [ ]:

plt.figure(figsize=(12,5))

plt.subplot(1,2,1)
plt.plot(train_acc,label="Train")
plt.plot(val_acc,label="Val")
plt.title("Accuracy")
plt.legend()

plt.subplot(1,2,2)
plt.plot(train_loss,label="Train")
plt.plot(val_loss,label="Val")
plt.title("Loss")
plt.legend()

plt.show()

## PREDICTIONS


In [ ]:

y_true, y_pred, y_prob = [], [], []

model.eval()
with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(DEVICE)
        outputs = model(images)
        probs = torch.softmax(outputs,1)

        y_true.extend(labels.numpy())
        y_pred.extend(outputs.argmax(1).cpu().numpy())
        y_prob.extend(probs.cpu().numpy())

y_true = np.array(y_true)
y_pred = np.array(y_pred)
y_prob = np.array(y_prob)

## CONFUSION MATRIX


In [ ]:

cm = confusion_matrix(y_true, y_pred)

plt.figure(figsize=(6,5))
sns.heatmap(cm, annot=True, fmt="d",
            xticklabels=CLASS_NAMES,
            yticklabels=CLASS_NAMES)
plt.xlabel("Predicted")
plt.ylabel("True")
plt.title("Confusion Matrix")
plt.show()



## ROC-AUC CURVE


In [ ]:

from sklearn.preprocessing import label_binarize
y_bin = label_binarize(y_true, classes=[0,1,2,3])

plt.figure(figsize=(7,6))

for i in range(NUM_CLASSES):
    fpr, tpr, _ = roc_curve(y_bin[:,i], y_prob[:,i])
    roc_auc = auc(fpr, tpr)
    plt.plot(fpr, tpr, label=f"{CLASS_NAMES[i]} AUC={roc_auc:.3f}")

plt.plot([0,1],[0,1],'k--')
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve")
plt.legend()
plt.show()